# 01 — Data preparation (Python analysis copy)

SQL Server `raw.online_retail` stays the source of truth.
This notebook builds a **working copy** in pandas. It does not create Silver/Gold tables,
does not load Excel, and does not drop rows.

Flags and code groups are **provisional** (from profiling). They are not approved Silver rules.

## 1. Read from SQL Server

In [1]:
import pandas as pd
import pyodbc
from decimal import Decimal, ROUND_HALF_UP

# Same working connection as the load script. Do not import that script
# (it inserts data).
server = r".\MSSQLSERVER02"
database = "RetailAnalytics"
driver = "ODBC Driver 18 for SQL Server"
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

read_sql = """
SELECT
    raw_row_id,
    InvoiceNo,
    StockCode,
    Description,
    Quantity,
    InvoiceDate,
    UnitPrice,
    CustomerID,
    Country,
    source_file,
    loaded_at
FROM raw.online_retail;
"""

connection = pyodbc.connect(connection_string)
try:
    df_raw = pd.read_sql(read_sql, connection)
finally:
    connection.close()

df = df_raw.copy()
print("Loaded rows:", len(df_raw))
print("Columns:", list(df_raw.columns))

C:\Users\Admin\AppData\Local\Temp\ipykernel_13556\979760040.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_raw = pd.read_sql(read_sql, connection)


Loaded rows: 541909
Columns: ['raw_row_id', 'InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'source_file', 'loaded_at']


## 2. Basic checks and types

In [2]:
print("shape:", df.shape)
print("\ndtypes:\n", df.dtypes)
print("\nmissing counts:\n", df.isna().sum())

shape: (541909, 11)

dtypes:
 raw_row_id              int64
InvoiceNo                 str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID                str
Country                   str
source_file               str
loaded_at      datetime64[us]
dtype: object

missing counts:
 raw_row_id          0
InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
source_file         0
loaded_at           0
dtype: int64


In [3]:
EXPECTED_ROWS = 541_909
assert len(df) == EXPECTED_ROWS, f"Row count {len(df)} != {EXPECTED_ROWS}"
assert df["raw_row_id"].notna().all()
assert df["raw_row_id"].nunique() == len(df)
print("Row count and unique non-null raw_row_id: OK")

Row count and unique non-null raw_row_id: OK


In [4]:
# Identifiers as pandas 'string' so missing CustomerID stays <NA>, not 'nan'.
for col in ["InvoiceNo", "StockCode", "CustomerID", "Country"]:
    df[col] = df[col].astype("string")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Original Description is left unchanged. This copy is trimmed; blanks become missing.
desc = df["Description"].astype("string").str.strip()
df["Description_clean"] = desc.mask(desc.eq(""), pd.NA)

print(df[["InvoiceNo", "StockCode", "CustomerID", "Description", "Description_clean"]].dtypes)

InvoiceNo            string
StockCode            string
CustomerID           string
Description             str
Description_clean    string
dtype: object


### Line value and money types

SQL Server `decimal` often arrives as Python `Decimal`. Pandas is easier with `float64`,
but floats cannot hold every penny exactly. We convert for analysis, then check totals
**to the nearest penny** (2 decimal places), not with `==` on raw floats.

Profiling (section 4.5): net **9,747,747.9340**, gross **10,666,684.5440**,
signed neg-qty **−896,812.4900**, negative-price **−22,124.1200**.

In [5]:
print("UnitPrice type sample:", type(df["UnitPrice"].iloc[0]))
df["UnitPrice"] = df["UnitPrice"].astype("float64")
df["line_value"] = df["Quantity"] * df["UnitPrice"]


def penny(x: float) -> Decimal:
    return Decimal(str(x)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


def penny_sum(series: pd.Series) -> Decimal:
    return penny(float(series.sum()))


SQL_NET = Decimal("9747747.9340")
SQL_GROSS = Decimal("10666684.5440")
SQL_SIGNED_NEG = Decimal("-896812.4900")
SQL_NEG_PRICE = Decimal("-22124.1200")

gross = df.loc[(df["Quantity"] > 0) & (df["UnitPrice"] > 0), "line_value"]
signed_neg = df.loc[(df["Quantity"] < 0) & (df["UnitPrice"] > 0), "line_value"]
neg_price = df.loc[df["UnitPrice"] < 0, "line_value"]

print("net pennies Python vs SQL:", penny_sum(df["line_value"]), penny(float(SQL_NET)))
print("gross pennies:", penny_sum(gross), penny(float(SQL_GROSS)))
print("signed_neg pennies:", penny_sum(signed_neg), penny(float(SQL_SIGNED_NEG)))
print("neg_price pennies:", penny_sum(neg_price), penny(float(SQL_NEG_PRICE)))

UnitPrice type sample: <class 'numpy.float64'>
net pennies Python vs SQL: 9747747.93 9747747.93
gross pennies: 10666684.54 10666684.54
signed_neg pennies: -896812.49 -896812.49
neg_price pennies: -22124.12 -22124.12


## 3. Flags and StockCode mapping

In [6]:
df["is_unidentified_customer"] = df["CustomerID"].isna()
df["is_missing_description"] = df["Description_clean"].isna()
# SQL C% is case-insensitive under the database collation.
df["is_c_prefix_invoice"] = df["InvoiceNo"].str.upper().str.startswith("C", na=False)
df["is_negative_quantity"] = df["Quantity"] < 0
df["is_zero_price"] = df["UnitPrice"] == 0
df["is_negative_price"] = df["UnitPrice"] < 0

print(df[
    [
        "is_unidentified_customer",
        "is_missing_description",
        "is_c_prefix_invoice",
        "is_negative_quantity",
        "is_zero_price",
        "is_negative_price",
    ]
].sum())

is_unidentified_customer    135080
is_missing_description        1454
is_c_prefix_invoice           9288
is_negative_quantity         10624
is_zero_price                 2515
is_negative_price                2
dtype: Int64


In [7]:
# Explicit map from profiling (4.1) plus C2 carriage and S samples.
# M stays manual/unresolved. Ordinary GIFT products are not listed.
STOCKCODE_GROUP = {
    "POST": "postage_carriage",
    "DOT": "postage_carriage",
    "C2": "postage_carriage",
    "AMAZONFEE": "fees_commission",
    "CRUK": "fees_commission",
    "BANK CHARGES": "fees_commission",
    "D": "discount",
    "M": "manual_unresolved",
    "S": "samples",
    "B": "bad_debt_adjustment",
}

df["code_group"] = df["StockCode"].map(STOCKCODE_GROUP).astype("string")
df["code_group"] = df["code_group"].fillna("merchandise_candidate")

# All observed Adjust bad debt lines (including the +11,062.06 B row).
adjust_bad_debt = df["Description"].astype("string").str.contains(
    "Adjust bad debt", case=False, na=False
)
df.loc[adjust_bad_debt, "code_group"] = "bad_debt_adjustment"

print(df["code_group"].value_counts(dropna=False))

code_group
merchandise_candidate    538998
postage_carriage           2110
manual_unresolved           571
fees_commission              87
discount                     77
samples                      63
bad_debt_adjustment           3
Name: count, dtype: Int64


## 4. Duplicates (keep every row)

In [8]:
BUSINESS_COLS = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country",
]

# dropna=False so NULL CustomerID groups like SQL GROUP BY (NULL = NULL).
g = df.groupby(BUSINESS_COLS, dropna=False, sort=False)
df["duplicate_group_size"] = g["raw_row_id"].transform("size")
df["duplicate_row_number"] = g["raw_row_id"].rank(method="first")
df["is_duplicate_extra"] = (df["duplicate_group_size"] > 1) & (df["duplicate_row_number"] > 1)

py_groups = int(df.loc[df["duplicate_group_size"] > 1].groupby(BUSINESS_COLS, dropna=False).ngroups)
py_in_groups = int((df["duplicate_group_size"] > 1).sum())
py_extras = int(df["is_duplicate_extra"].sum())

SQL_GROUPS, SQL_IN_GROUPS, SQL_EXTRAS = 4879, 10147, 5268
print("Python groups / in-groups / extras:", py_groups, py_in_groups, py_extras)
print("SQL    groups / in-groups / extras:", SQL_GROUPS, SQL_IN_GROUPS, SQL_EXTRAS)
print("Match SQL:", (py_groups, py_in_groups, py_extras) == (SQL_GROUPS, SQL_IN_GROUPS, SQL_EXTRAS))
print(
    "Note: SQL used Chinese_PRC_CI_AS (case-insensitive). "
    "Python string equality is case-sensitive. We did not lowercase to force a match."
)

Python groups / in-groups / extras: 4879 10147 5268
SQL    groups / in-groups / extras: 4879 10147 5268
Match SQL: True
Note: SQL used Chinese_PRC_CI_AS (case-insensitive). Python string equality is case-sensitive. We did not lowercase to force a match.


In [9]:
keep = ~df["is_duplicate_extra"]
print("All rows  net / gross pennies:", penny_sum(df["line_value"]), penny_sum(gross))
print(
    "No extras net / gross pennies:",
    penny_sum(df.loc[keep, "line_value"]),
    penny_sum(df.loc[keep & (df["Quantity"] > 0) & (df["UnitPrice"] > 0), "line_value"]),
)
print("Main dataframe still has all rows:", len(df) == EXPECTED_ROWS)

All rows  net / gross pennies: 9747747.93 10666684.54
No extras net / gross pennies: 9726006.95 10642110.80
Main dataframe still has all rows: True


## 5. Time fields (no RFM scores yet)

In [10]:
df["invoice_month"] = df["InvoiceDate"].dt.to_period("M")
max_ts = df["InvoiceDate"].max()
final_month = max_ts.to_period("M")
df["is_final_partial_month"] = df["invoice_month"] == final_month

# One timestamp per invoice: earliest line time (keeps line InvoiceDate as-is).
df["order_timestamp"] = df.groupby("InvoiceNo", dropna=False)["InvoiceDate"].transform("min")

inv = df.groupby("InvoiceNo", dropna=False).agg(
    min_ts=("InvoiceDate", "min"),
    max_ts=("InvoiceDate", "max"),
)
cross_date = (inv["min_ts"].dt.normalize() != inv["max_ts"].dt.normalize()).sum()
cross_month = (inv["min_ts"].dt.to_period("M") != inv["max_ts"].dt.to_period("M")).sum()
multi_ts = (inv["min_ts"] != inv["max_ts"]).sum()

rfm_reference_date = (max_ts.normalize() + pd.Timedelta(days=1)).date()

print("max InvoiceDate:", max_ts)
print("RFM reference date (day after max):", rfm_reference_date)
print("Invoices with >1 timestamp:", int(multi_ts))
print("Invoices crossing calendar dates:", int(cross_date))
print("Invoices crossing calendar months:", int(cross_month))
print("Lines in final partial month:", int(df["is_final_partial_month"].sum()))

max InvoiceDate: 2011-12-09 12:50:00
RFM reference date (day after max): 2011-12-10
Invoices with >1 timestamp: 43
Invoices crossing calendar dates: 0
Invoices crossing calendar months: 0
Lines in final partial month: 25525


## 6. Analysis masks (rows are not deleted)

Negative-value merchandise lines are **not** labelled as confirmed cash refunds.
Duplicate extras stay in the baseline masks.

In [11]:
is_merch = df["code_group"] == "merchandise_candidate"

mask_merchandise_sale = (
    is_merch
    & (df["Quantity"] > 0)
    & (df["UnitPrice"] > 0)
    & ~df["is_c_prefix_invoice"]
)
mask_merchandise_negative_value = (
    is_merch & (df["Quantity"] < 0) & (df["UnitPrice"] > 0)
)
mask_identified_customer_purchase = mask_merchandise_sale & df["CustomerID"].notna()

print("merchandise sale rows / value pennies:", int(mask_merchandise_sale.sum()), penny_sum(df.loc[mask_merchandise_sale, "line_value"]))
print(
    "merchandise negative-value rows / value pennies:",
    int(mask_merchandise_negative_value.sum()),
    penny_sum(df.loc[mask_merchandise_negative_value, "line_value"]),
)
print(
    "identified-customer purchase rows / value pennies:",
    int(mask_identified_customer_purchase.sum()),
    penny_sum(df.loc[mask_identified_customer_purchase, "line_value"]),
)

merchandise sale rows / value pennies: 527793 10272121.42
merchandise negative-value rows / value pennies: 8704 -478724.18
identified-customer purchase rows / value pennies: 396340 8761066.65


## 7. Validate and summarize

In [12]:
assert len(df) == EXPECTED_ROWS
assert df["raw_row_id"].nunique() == EXPECTED_ROWS
assert set(df_raw.columns).issubset(df.columns)
assert df["Description"].astype("string").equals(df_raw["Description"].astype("string"))

print("Coverage OK: 541,909 rows, unique raw_row_id, original Description untouched.")
print("\nFlag counts:")
print(df[["is_unidentified_customer", "is_missing_description", "is_c_prefix_invoice",
         "is_negative_quantity", "is_zero_price", "is_negative_price",
         "is_duplicate_extra", "is_final_partial_month"]].sum())
print("\ncode_group counts:")
print(df["code_group"].value_counts())

Coverage OK: 541,909 rows, unique raw_row_id, original Description untouched.

Flag counts:
is_unidentified_customer    135080
is_missing_description        1454
is_c_prefix_invoice           9288
is_negative_quantity         10624
is_zero_price                 2515
is_negative_price                2
is_duplicate_extra            5268
is_final_partial_month       25525
dtype: Int64

code_group counts:
code_group
merchandise_candidate    538998
postage_carriage           2110
manual_unresolved           571
fees_commission              87
discount                     77
samples                      63
bad_debt_adjustment           3
Name: count, dtype: Int64


### What this notebook did

- **Changed (in `df` only):** types, `Description_clean`, `line_value`, flags, `code_group`,
  duplicate fields, month / partial-month / `order_timestamp` / RFM reference date, analysis masks.
- **Untouched:** SQL Raw table; `df_raw`; original `Description`; every row and `raw_row_id`.
- **Provisional:** code groups, merchandise masks, duplicate extras kept in baseline.
- **Ready next (not done here):** monthly revenue, country mix, and top identified customers
  using the masks — with Dec 2011 treated as a partial month and RFM snapshot date stored,
  scores not yet computed.